In [1]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"   # can also be "jax" or "torch"

import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
import keras
from keras import layers, ops
from keras import regularizers

In [2]:
# ─── Hyperparameters ───────────────────────────────────────────────

IMAGE_SIZE     = 224          # we'll resize images to this
PATCH_SIZE     = 16           # 16×16 patches → 196 patches for 224×224
NUM_PATCHES    = (IMAGE_SIZE // PATCH_SIZE) ** 2

EMB_DIM        = 256          # patch embedding dimension
NUM_HEADS      = 8
FF_DIM         = 512          # feed-forward hidden size
DROPOUT_RATE   = 0.1

NUM_LAYERS     = 8            # number of Transformer encoder layers
NUM_CLASSES    = 100          # CIFAR-100

BATCH_SIZE     = 128
EPOCHS         = 60           # increase if you have more compute
LEARNING_RATE  = 3e-4


In [3]:
# ─── Data ──────────────────────────────────────────────────────────

(x_train, y_train), (x_test, y_test) = keras.datasets.cifar100.load_data()

# For quicker debugging you can use only 10 classes:
# idx = y_train.flatten() < 10
# x_train, y_train = x_train[idx], y_train[idx]
# idx = y_test.flatten() < 10
# x_test, y_test   = x_test[idx], y_test[idx]
# NUM_CLASSES = 10

print(f"Training samples: {x_train.shape[0]}   Test samples: {x_test.shape[0]}")

# Convert labels to categorical (one-hot)
y_train = keras.utils.to_categorical(y_train, NUM_CLASSES)
y_test  = keras.utils.to_categorical(y_test,  NUM_CLASSES)

# Simple augmentation pipeline
data_augmentation = keras.Sequential([
    layers.Normalization(),                 # will be adapted later
    layers.Resizing(IMAGE_SIZE, IMAGE_SIZE),
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.2),
], name="data_augmentation")

# Important: adapt normalization on training data only
data_augmentation.layers[0].adapt(x_train)



Training samples: 50000   Test samples: 10000


In [4]:
# ─── Patch Embedding ───────────────────────────────────────────────

class PatchEncoder(layers.Layer):
    def __init__(self, num_patches, embed_dim, **kwargs):
        super().__init__(**kwargs)
        self.num_patches = num_patches
        self.projection = layers.Dense(embed_dim)
        self.position_embedding = layers.Embedding(
            input_dim=num_patches, output_dim=embed_dim
        )

    def call(self, patch):
        positions = ops.arange(start=0, stop=self.num_patches, step=1)
        encoded = self.projection(patch) + self.position_embedding(positions)
        return encoded

    def get_config(self):
        config = super().get_config()
        config.update({"num_patches": self.num_patches})
        return config


def create_vit_classifier():
    inputs = keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3))

    # Augmentation (only applied during training)
    augmented = data_augmentation(inputs)

    # Patch splitting & flattening
    patches = layers.Conv2D(
        filters=EMB_DIM,
        kernel_size=PATCH_SIZE,
        strides=PATCH_SIZE,
        padding="valid",
        name="patch_projection"
    )(augmented)                               # → (None, 14, 14, EMB_DIM)

    patches = layers.Reshape(
        target_shape=(NUM_PATCHES, EMB_DIM)
    )(patches)                                 # → (None, 196, EMB_DIM)

    # Add position embedding
    encoded_patches = PatchEncoder(NUM_PATCHES, EMB_DIM)(patches)

    # Dropout after embedding
    x = layers.Dropout(DROPOUT_RATE)(encoded_patches)

    # Transformer blocks
    for _ in range(NUM_LAYERS):
        # Layer normalization 1
        x1 = layers.LayerNormalization(epsilon=1e-6)(x)

        # Multi-head attention
        attention_output = layers.MultiHeadAttention(
            num_heads=NUM_HEADS,
            key_dim=EMB_DIM // NUM_HEADS,
            dropout=DROPOUT_RATE
        )(x1, x1)

        # Skip connection
        x2 = layers.Add()([attention_output, x])

        # Layer normalization 2
        x3 = layers.LayerNormalization(epsilon=1e-6)(x2)

        # MLP (feed-forward)
        x3 = layers.Dense(FF_DIM, activation="gelu")(x3)
        x3 = layers.Dense(EMB_DIM)(x3)
        x3 = layers.Dropout(DROPOUT_RATE)(x3)

        # Second skip connection
        x = layers.Add()([x3, x2])

    # Final representation → global average pooling / cls token style
    representation = layers.LayerNormalization(epsilon=1e-6)(x)
    representation = layers.GlobalAveragePooling1D()(representation)

    # Head
    features = layers.Dropout(0.3)(representation)
    features = layers.Dense(FF_DIM // 2, activation="gelu")(features)
    features = layers.Dropout(0.3)(features)

    logits = layers.Dense(NUM_CLASSES)(features)

    model = keras.Model(inputs=inputs, outputs=logits)
    return model

In [5]:
# ─── Build & Compile ───────────────────────────────────────────────

model = create_vit_classifier()
model.summary()

optimizer = keras.optimizers.AdamW(
    learning_rate=LEARNING_RATE,
    weight_decay=0.05,           # stronger regularization helps ViTs
    global_clipnorm=1.0
)

model.compile(
    optimizer=optimizer,
    loss=keras.losses.CategoricalCrossentropy(from_logits=True),
    metrics=[
        keras.metrics.CategoricalAccuracy(name="acc"),
        keras.metrics.TopKCategoricalAccuracy(k=5, name="top5")
    ]
)


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ data_augmentation   │ (None, 224, 224,  │          7 │ input_layer[0][0] │
│ (Sequential)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ patch_projection    │ (None, 14, 14,    │    196,864 │ data_augmentatio… │
│ (Conv2D)            │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape (Reshape)   │ (None, 196, 256)  │          0 │ patch_projection… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ patch_encoder       │ (None, 196, 256)  │    115,968 │ reshape[0][0]     │
│ (PatchEncoder)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 196, 256)  │          0 │ patch_encoder[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 196, 256)  │        512 │ dropout[0][0]     │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 196, 256)  │    263,168 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 196, 256)  │          0 │ multi_head_atten… │
│                     │                   │            │ dropout[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 196, 256)  │        512 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 196, 512)  │    131,584 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 196, 256)  │    131,328 │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_2 (Dropout) │ (None, 196, 256)  │          0 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 196, 256)  │          0 │ dropout_2[0][0],  │
│                     │                   │            │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 196, 256)  │        512 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 196, 256)  │    263,168 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 196, 256)  │          0 │ multi_head_atten… │
│                     │                   │            │ add_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 196, 256)  │        512 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 4,621,675 (17.63 MB)

 Trainable params: 4,621,668 (17.63 MB)

 Non-trainable params: 7 (32.00 B)

In [ ]:
# ─── Train ─────────────────────────────────────────────────────────

callbacks = [
    keras.callbacks.ModelCheckpoint(
        "vit_best.keras", save_best_only=True, monitor="val_acc"
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_acc", patience=12, restore_best_weights=True
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6
    ),
    keras.callbacks.TensorBoard(log_dir="logs/vit")
]

def preprocess(image, label):
    image = tf.cast(image, tf.float32)          # important: cast before resize
    image = tf.image.resize(image, [224, 224])
    image = image / 255.0                       # or use tf.keras.applications.vit.preprocess_input(image)
    # Optional: add light augmentation here
    # image = tf.image.random_flip_left_right(image)
    # image = tf.image.random_brightness(image, 0.2)
    return image, label

# Build dataset pipeline
BATCH_SIZE = 32   # ← start small (16–64), increase later if GPU allows

train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train)) \
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE) \
    .shuffle(buffer_size=5000) \
    .batch(BATCH_SIZE) \
    .prefetch(tf.data.AUTOTUNE)

# Validation split example (10%)
val_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train)) \
    .take(int(0.1 * len(x_train))) \
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE) \
    .batch(BATCH_SIZE) \
    .prefetch(tf.data.AUTOTUNE)


history = model.fit(
    train_ds,
    validation_data=val_ds,          # or keep validation_split=0.1 if you prefer (but datasets are better)
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)


Epoch 1/60
 224/1563 ━━━━━━━━━━━━━━━━━━━━ 15:55 714ms/step - acc: 0.0092 - loss: 4.7379 - top5: 0.0474

In [ ]:

# ─── Evaluate ──────────────────────────────────────────────────────

test_loss, test_acc, test_top5 = model.evaluate(x_test, y_test)
print(f"\nTest accuracy: {test_acc:.4f}    Top-5: {test_top5:.4f}")

# Optional: plot training curves
plt.figure(figsize=(10,4))
plt.subplot(1,2,1)
plt.plot(history.history["acc"], label="train acc")
plt.plot(history.history["val_acc"], label="val acc")
plt.legend(); plt.title("Accuracy")
plt.subplot(1,2,2)
plt.plot(history.history["loss"], label="train loss")
plt.plot(history.history["val_loss"], label="val loss")
plt.legend(); plt.title("Loss")
plt.show()